# Task 3 — Streaming Auto Loader: landing -> bronze (Delta)

In [0]:
# Configuration

from pyspark.sql.functions import current_timestamp, current_date

CATALOG    = "dbr_dev"
STORAGE_ACCOUNT = "dlspl21databricks"
CONTAINER  = "gabrielajaniszews786"
SCHEMA     = "gabrielajaniszews786_bronze"
BASE       = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
CHECKPOINT = f"{BASE}/_checkpoint/sensor_data" # Checkpoint
TARGET     = f"{CATALOG}.{SCHEMA}.sensor_data"

### Setting up stream consumer using Kafka

Streaming source - Event Hub read as Kafka
Event Hub exposes a Kafka-compatible endpoint (port 9093).

In [0]:
EH_NAMESPACE = "evhpl24databricks"
EH_NAME = "gabrielajaniszews786_eventhub"
EH_CONN_STR = dbutils.secrets.get("default2", "eventhub-con-str-gabriela")

BOOTSTRAP = f"{EH_NAMESPACE}.servicebus.windows.net:9093"
JAAS = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{EH_CONN_STR}";'

raw = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", EH_NAME)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", JAAS)
    .option("startingOffsets", "earliest")   # Reading from the earliest available data
    .load())


## Parsing the payload and attach metadata
The message body arrives as a binary "value". It is cast to a string and JSON file is unpacked against a fixed schema.

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Explicit schema
event_schema = (StructType()
    .add("event_id", StringType())
    .add("schema_version", IntegerType())
    .add("site_id", StringType())
    .add("site_name", StringType())
    .add("country", StringType())
    .add("bidding_zone", StringType())
    .add("timestamp_utc", StringType())
    .add("consumption_kwh", DoubleType())
    .add("avg_power_kw", DoubleType())
    .add("pue", DoubleType())
    .add("reading_interval_s", IntegerType())
)

parsed = (raw
    .select(
        from_json(col("value").cast("string"), event_schema).alias("e"), # binary value > string > JSON
        col("partition"), col("offset"), # Event Hub metadata (where the record came from)
        col("timestamp").alias("enqueued_ts")) # when it landed in Event Hub
    .select("e.*", "partition", "offset", "enqueued_ts")  # explode the "e" struct into columns
    .withColumn("ingestion_ts", current_timestamp())) # when the data was ingested



## Write to the bronze layer (Delta)
The stream writes to a Delta table. The `availableNow` trigger processes the
backlog and stops (it doesn't run 24/7). The checkpoint guarantees no record is read twice.

In [0]:
query = (parsed.writeStream
   .format("delta")
   .option("checkpointLocation", CHECKPOINT)
   .option("mergeSchema","true")  
   .trigger(availableNow=True)                
   .toTable(TARGET))

## Verify — how many records landed

In [0]:
spark.table(TARGET).count()

## Preview the table contents

In [0]:
%sql

SELECT *
FROM gabrielajaniszews786_bronze.sensor_data

In [0]:
for q in spark.streams.active:
    q.stop()

In [0]:
# Checking active streams

for q in spark.streams.active:
    print(q.name, q.id, q.status)